# 05 · Your First AI Agent 🤖

Yesterday Uncle's brain was a pile of `if` statements. Today we give him a real brain.

An **agent** = a **model** (the AI) + **instructions** (its personality and rules) + **tools** (things it can do).

We use the **OpenAI Agents SDK** (`pip install openai-agents`).

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

MODEL = os.environ.get("OPENAI_MODEL") or "gpt-6-luna"
print("Ready! Using", MODEL)

## 1. Create an agent

In [ ]:
from agents import Agent, Runner

uncle = Agent(
    name="Kopi Uncle",
    instructions="You are a friendly kopitiam uncle in Singapore. Speak light Singlish. Keep replies short.",
    model=MODEL,
)

In [ ]:
# ask it something - `await` means "wait for the AI to reply"
result = await Runner.run(uncle, "Uncle, what drink you recommend for a hot afternoon?")

print(result.final_output)

> 💡 **Why `await`?** Notebooks can wait for slow things (like the internet) with `await`.
> In a normal `.py` file we use `Runner.run_sync(...)` instead — you'll see it in `main.py`.

## 2. Instructions = personality
Same question, different instructions. Run it a few times — the AI doesn't answer the same way twice!

In [ ]:
grumpy = Agent(
    name="Grumpy Uncle",
    instructions="You are a grumpy but lovable kopitiam uncle. Complain a lot, then help anyway. Max 2 lines.",
    model=MODEL,
)

result = await Runner.run(grumpy, "Uncle, what drink you recommend for a hot afternoon?")
print(result.final_output)

## 3. Thinking speed
`gpt-6-luna` can *reason* (think) before it answers. More thinking = smarter but slower.
For a chat bot, **low** is usually best.

In [ ]:
import time

from agents import ModelSettings

for effort in ["low", "medium"]:
    agent = Agent(
        name="Kopi Uncle",
        instructions="You are a friendly kopitiam uncle. Keep replies short.",
        model=MODEL,
        model_settings=ModelSettings(reasoning={"effort": effort}),
    )
    start = time.time()
    result = await Runner.run(agent, "Kopi or teh is better? Why?")
    print(f"[{effort}] {time.time() - start:.1f}s -> {result.final_output}\n")

## 4. The problem: it makes things up 😬

In [ ]:
result = await Runner.run(uncle, "How much is one kopi-o peng and two kaya toast?")

print(result.final_output)

Uncle just **invented** a price. He has never seen our menu!

**Next notebook:** we give him **tools** — so he can look up the real menu and calculate the real bill.

## 5. Swap the brain
Remember `reply()` from Day 1? Here it is with an AI brain. Same input, same output — much smarter inside.

In [ ]:
async def reply(text, name):
    result = await Runner.run(uncle, f"{name} says: {text}")
    return result.final_output


print(await reply("can I have a kopi but less sweet?", "Mei Ling"))

## ✏️ Exercises
1. Create your own agent with a different personality: a **hawker auntie**, an **MRT station guide**, a **Singlish tutor**...
2. Add a rule to the instructions so it **refuses** anything not about food. Test it with *"help me with my homework"*.
3. Try `effort="none"` — is it faster? Is the answer worse?

In [ ]:
# your code here